In [1]:
!pip install conllu
!pip install unsloth==2026.4.2

from google.colab import drive
drive.mount('/content/drive')
"""
#!unzip /content/drive/MyDrive/parseme.zip -d /content/drive/MyDrive/parseme
#!mkdir -p /content/drive/MyDrive/parseme_data

!for f in /content/drive/MyDrive/parseme/*.tgz; do \
  tar -xzf "$f" -C /content/drive/MyDrive/parseme_data; \
done
"""
import os
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
import torch
import gc
import re
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
torch.use_deterministic_algorithms(True)

from transformers import set_seed


import pandas as pd
import conllu
from unsloth import FastLanguageModel
from datasets import Dataset
from transformers import TrainingArguments
import pandas as pd
from transformers import EarlyStoppingCallback
from trl import SFTTrainer, SFTConfig
###
#code adapted from Google Search's Gemini @ google.com



def file_to_parse(file_path):
    with open(file_path, 'r', encoding = 'utf-8') as f:
        data = f.read()

    sentences = conllu.parse(data)
    return sentences
###


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.8/55.8 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.0/63.0 MB 43.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 44.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 70.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 33.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 58.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 421.9/421.9 kB 58.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 162.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 64.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.2/185.2 kB 31.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 60.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 159.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 225

/tmp/ipykernel_4302/190125993.py:28: UserWarning: WARNING: Unsloth should be imported before [transformers, peft] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from unsloth import FastLanguageModel


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: Your Flash Attention 2 installation seems to be broken. Using Xformers instead. No performance changes will be seen.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [2]:
import re
def find_compounds(sentences, add_to_vmwe = False, categorized = True):

  def recursive_compound_labeling(sentence, index, label, value):

    if sentence[index]['head'] != None:
      compound = sentence[index]['head']-1
    else:
      if sentence[index]['parseme:mwe'] == '*':
          sentence[index]['parseme:mwe'] = str(value) + label
      else:
        #something is wrong
        pass
      return sentence, [index]


    if 'compound' not in sentence[index]['deprel']:
      if 'COMP' not in sentence[index]['parseme:mwe']:
        if sentence[index]['parseme:mwe'] == '*':
          sentence[index]['parseme:mwe'] = str(value) + label
        else:
          #something is wrong
          pass
      return sentence, [index]



    elif 'compound' in sentence[index]['deprel'] and sentence[compound]['parseme:mwe'] != '*':
      value = sentence[compound]['parseme:mwe'].split(':')[0]

      if sentence[index]['parseme:mwe'] == '*':
        sentence[index]['parseme:mwe'] = str(value)
      else:
        pass
        #somethings wrong
      return sentence, [index, compound]

    else:
      sentence = recursive_compound_labeling(sentence,compound,label,value)
      return recursive_compound_labeling(sentence[0], index,label, value)[0], sentence[1] + [index]



  if add_to_vmwe == True:
    pass
  else:
    #can't be in more than one compound using PARSEME
    i = 0
    while i < len(sentences):
      skip = []
      count = 1
      sentence_ok = True

      for j in range(len(sentences[i])):
        if sentences[i][j]['id'] != (j+1):
          sentence_ok = False
      if sentence_ok == True:
        for j in range(len(sentences[i])):
          sentences[i][j]['parseme:mwe'] = '*'
        for j in range(len(sentences[i])):

          if 'COMP' not in sentences[i][j]['parseme:mwe']:
            if 'compound' in sentences[i][j]['deprel']:

                #recursive compound detection
              if j in skip:
                pass
              else:
                label = ':COMP'
                #recursive labeling
                #print(i)
                if categorized == True:
                  typ = sentences[i][j]['deprel'].split(':')
                  if len(typ) > 1:
                    label = ':COMP-' + typ[1].upper()
                #print(i)
                e = recursive_compound_labeling(sentences[i],j,label,count)

                sentences[i] = e[0]
                #print(len(e[0]))
                skip = skip + e[1]
                count = count+1
      else:
        del sentences[i]
        i = i-1
      i = i+1


  return sentences

def parse_to_dict(sentences):
  inputs = []
  labels = []
  for i in sentences:
    inputt = []
    label = []
    for j in range(len(i)):
        inputt.append(i[j]['form'])
        label.append(i[j]['parseme:mwe'])
    inputs.append(inputt)
    labels.append(label)
  dictionary = {'sentence': inputs, 'label': labels}
  return dictionary


In [3]:
def find_spans(labels, usage = 'full'):
  #assumes labels are like 1:VID, 1, etc
  #usage = 'full' means it outputs everything in between in a non consecutive span
  #this method works for one sentence inputs
  spans = {}
  if usage == 'full':
    for i in range(len(labels)):
      if labels[i] != '*':
        split = labels[i].split(';')
        for j in split:
          try:
            int(j)
            ### for compounds only
            if str(j) not in spans:
              spans[str(j)] = [i]
            else:
              if isinstance(spans[str(j)][len(spans[str(j)])-1],str):
                if len(spans[str(j)]) == 2:
                  spans[str(j)].insert(1,i+1)
                else:
                  spans[str(j)][1] = i+1
                #spans[str(j)][0:1] = spans[str(j)][0:1].sort()
              else:
                if len(spans[str(j)]) == 1:
                  spans[str(j)].append(i+1)
                else:
                  spans[str(j)][1] = i+1
                #spans[str(j)][0:1] = spans[str(j)][0:1].sort()

          except ValueError:
            a = j.split(':')
            if a[0] in spans:
              if len(spans[a[0]]) == 1:
                spans[a[0]].append(i+1)
              else:
                spans[a[0]][1] = i+1
              #spans[a[0]][0:1] = spans[a[0]][0:1].sort()
              spans[a[0]].append(a[1])
            else:
              spans[a[0]] = [i, a[1]]
  else:
    for i in range(len(labels)):
      if labels[i] != '*':
        split = labels[i].split(';')
        for j in split:
          try:
            int(j)
            if str(j) not in spans:
              spans[str(j)] = [i]
            else:
              if isinstance(spans[str(j)][len(spans[str(j)])-1],str):
                spans[str(j)].insert(len(spans[str(j)])-1,i)
                #spans[str(j)][0:len(spans[str(j)])-1] = spans[str(j)][0:len(spans[str(j)])-1].sort()
              else:
                spans[str(j)].append(i)
                #spans[str(j)] = spans[str(j)].sort()
          except ValueError:
            a = j.split(':')
            if a[0] not in spans:
              spans[a[0]] = [i, a[1]]
            else:
              spans[a[0]].append(i)
              spans[a[0]].append(a[1])

  if usage == 'nominal_only':
    for m in spans:
      if spans[m][len(spans[m])-1] == 'COMP':
        if len(spans[m]) >= 3:
          aa = sorted(spans[m][0:len(spans[m])-1])
          new_list = []
          for n in range(aa[0],aa[len(aa)-1]+1):
            new_list.append(n)
          new_list.append(spans[m][len(spans[m])-1])
          spans[m] = new_list

  #print(spans)
  return spans

pr_compound = """/no_think 复合词是由多个实义语素组成，并表达一个整体意义的词或表达。包括但不限于名词性复合词和短语动词。复合词的意义可以分为三种：完全组合、半组合和非组合。

请按照以下格式输出：复合词类型（如COMP,COMP-PRT,等），后接个冒号，再写出该复合词。

没有的话写'No compound found.' 如果有多个复合词，请每个单独占一行。如果有嵌套，只输出最高层、还能表示一个整体意义的复合词。

"""

pr_vmwe = """/no_think 请识别并标注句子中的所有VMWE（动词性多词表达）。

VMWE类型包括但不限于：动词性习语、轻动词结构、固有反身动词、动词-小品词结构、多动词结构以及固有介词动词。

你的标注应包含VMWE类型，后跟一个冒号，再写出该VMWE。如果有多个VMWE，请每个单独占一行。如果没有VMWE，请输出'No MWE found.'

"""

pr_vmwe_fewshot = """/no_think 请识别并标注句子中的所有VMWE（动词性多词表达）。

VMWE类型包括但不限于：动词性习语、轻动词结构、固有反身动词、动词-小品词结构、多动词结构以及固有介词动词。

各类别的VMWE示例如下：

MVC: 試試看, 分成
LVC.full: 做改造
LVC.cause: 帶來麻煩
VPC.semi: 做出來, 找到
VID: 風雲驟起, 千變萬化

你的标注应包含VMWE类型，后跟一个冒号，再写出该VMWE。如果有多个VMWE，请每个单独占一行。如果没有VMWE，请输出'No MWE found.'

"""

pr_compound_fewshot = """/no_think 复合词是由多个实义语素组成，并表达一个整体意义的词或表达。包括但不限于名词性复合词和短语动词。复合词的意义可以分为三种：完全组合、半组合和非组合。

例子如下：

COMP: 安全帽, 近代文学, 夜班, 星星，问题，重型火砲關鍵部件
COMP-EXT: 記得，表現得

请按照以下格式输出：复合词类型（如COMP,COMP-PRT,等），后接个冒号，再写出该复合词。

没有的话写'No compound found.' 如果有多个复合词，请每个单独占一行。如果有嵌套，只输出最高层、还能表示一个整体意义的复合词。

"""



def sentence_to_text(sentences, prompt = pr_vmwe, usage = 'full', compound = False, spaces = True):
  #usage = full for full spans, anything else for only the labeled MWE
  prompts = {'prompt': [], 'sentence': [], 'output': []}
  a = parse_to_dict(sentences)

  for i in range(len(a['sentence'])):
    response = ''
    if spaces == True:
      sentence = ' '.join(a['sentence'][i])
    else:
      sentence = ''.join(a['sentence'][i])
    prompts['prompt'].append(prompt)
    prompts['sentence'].append(sentence)
    #print(i)
    spans = find_spans(a['label'][i], usage)
    if usage=='full':
      for j in spans:
        #print(spans[j])
        b = a['sentence'][i][spans[j][0]:spans[j][1]]
        if spaces == True:
          b = ' '.join(b)
        else:
          b = ''.join(b)
        response = response + spans[j][2] + ': ' + b + '\n'
    else:
      for j in spans:
        b = [a['sentence'][i][k] for k in spans[j][0:len(spans[j])-1]]
        if spaces == True:
          b = ' '.join(b)
        else:
          b = ''.join(b)
        response = response + spans[j][len(spans[j])-1] + ': ' + b + '\n'
    response = response[0:len(response)-1] #get rid of the last newline
    if response == '' and compound == True:
      response = 'No compound found.'
    elif response == '':
      response = 'No MWE found.'
    prompts['output'].append(response)
  return prompts



In [4]:
import random
from peft import PeftModel, PeftConfig
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import concatenate_datasets
import numpy as np
def inference(start, end, model_type, compound_or_vmwe, language, test_language):
  #language is lowercase like 'en'
  #test language is uppercae like 'EN'
  if test_language in ['ZH', 'HI', 'AR']:
    spaces = False
  else:
    spaces = True

  path = '/content/drive/MyDrive/capstone/'
  file_path = path+'parseme_data/'+test_language+'/dev.cupt'

  sentences = file_to_parse(file_path)

  e = 0
  while e < len(sentences):
    if len(sentences[e]) > 125:
      sentences.pop(e)
      e = e-1
    e = e+1


  if 'compound' in compound_or_vmwe:
    sentences = find_compounds(sentences, categorized = True)
    if '0shot' in compound_or_vmwe:


      if test_language in ['EN', 'ES']:
        data = sentence_to_text(sentences, prompt = pr_compound, usage = 'nominal_only', compound = True, spaces = spaces)
      else:
        data = sentence_to_text(sentences, prompt = pr_compound, usage = 'full', compound = True, spaces = spaces)
    else:

      if test_language in ['EN', 'ES']:
        data = sentence_to_text(sentences, prompt = pr_compound_fewshot, usage = 'nominal_only', compound = True, spaces = spaces)
      else:
        data = sentence_to_text(sentences, prompt = pr_compound_fewshot, usage = 'full', compound = True, spaces = spaces)
  else:
    if '0shot' in compound_or_vmwe:
      data = sentence_to_text(sentences, prompt = pr_vmwe, usage = 'not_full', compound = False, spaces = spaces)
    else:
      data = sentence_to_text(sentences, prompt = pr_vmwe_fewshot, usage = 'not_full', compound = False, spaces = spaces)

  data = Dataset.from_dict(data)

  if 'vmwe' in compound_or_vmwe:
    def rebalance(unbalanced_data, seed):
    ##adapted from gpt code
      positive = unbalanced_data.filter(lambda x: x['output'] != 'No MWE found.')
      negative = unbalanced_data.filter(lambda x: x['output'] == 'No MWE found.')

      if len(positive) < 100:
        if len(negative) >= len(positive):
          a = len(positive)
        else:
          a = len(negative)
      elif len(negative) < 100:
        a = len(negative)
      else:
        a = 100

      rng = np.random.default_rng(seed)
      select_pos = rng.choice(len(positive), size = a, replace = False)
      positive = positive.select(select_pos.tolist())

      rng = np.random.default_rng(seed)
      select_neg = rng.choice(len(negative), size = a, replace = False)
      negative = negative.select(select_neg.tolist())

      dataset = concatenate_datasets([positive, negative])
      dataset = dataset.shuffle(seed=seed)

      return dataset
  elif 'compound' in compound_or_vmwe:
    def rebalance(unbalanced_data, seed):
    ##adapted from gpt code
      positive = unbalanced_data.filter(lambda x: x['output'] != 'No compound found.')
      negative = unbalanced_data.filter(lambda x: x['output'] == 'No compound found.')

      if len(positive) < 100:
        if len(negative) >= len(positive):
          a = len(positive)
        else:
          a = len(negative)
      elif len(negative) < 100:
        a = len(negative)
      else:
        a = 100

      rng = np.random.default_rng(seed)
      select_pos = rng.choice(len(positive), size = a, replace = False)
      positive = positive.select(select_pos.tolist())

      rng = np.random.default_rng(seed)
      select_neg = rng.choice(len(negative), size = a, replace = False)
      negative = negative.select(select_neg.tolist())

      dataset = concatenate_datasets([positive, negative])
      dataset = dataset.shuffle(seed=seed)
      ##
      return dataset

  data = rebalance(data,42)
  print(data['sentence'][0])
  for d in range(start,end):
      set_seed(d)


      ###adapted from GPT code

      model, tokenizer = FastLanguageModel.from_pretrained('unsloth/Meta-Llama-3.1-8B-Instruct',
                                                           load_in_4bit = False)
      #model.load_adapter(path+'models/'+model_type+'/'+compound_or_vmwe+'_'+language+'_'+str(d))
      """
      model = AutoModelForCausalLM.from_pretrained(model_name)
      tokenizer = AutoTokenizer.from_pretrained(model_name)
      config = PeftConfig.from_pretrained(path+'models/'+model_type+'/'+compound_or_vmwe+'_'+language+'_'+str(d))
      model = PeftModel(model,
                        path+'models/'+model_type+'/'+compound_or_vmwe+'_'+language+'_'+str(d), config=config)
      """
      print(1)
      model.eval()
      if tokenizer.pad_token is None:
          tokenizer.pad_token = tokenizer.eos_token

      model.config.pad_token_id = tokenizer.pad_token_id

      def format_test(instance):
        return {
            "messages": [
                {"role": "user", "content": instance['prompt'] + "\n\n句子如下：\n" + instance['sentence']}
            ]
        }

      def template_test(instance, tokenizer, tk = False):
        return{
            "text": tokenizer.apply_chat_template(
                instance["messages"],
                add_generation_prompt = True,
                tokenize = tk,
                return_tensors = "pt"
            )
        }

      ###

      dataset = data.map(format_test)
      dataset = dataset.map(template_test, fn_kwargs={'tokenizer': tokenizer})
      ###
      test = {'sentence': [], 'full_output': [], 'output': [], 'label': []}
      for i in range(len(dataset['text'])):
        ###chatgpt code
        inputs = tokenizer(dataset['text'][i], return_tensors = "pt", padding = True, truncation = False)
        outputs = model.generate(inputs['input_ids'].cuda(),
                                attention_mask = inputs['attention_mask'].cuda(),
                                max_new_tokens = 100,
                                temperature = 0,
                                do_sample = False,
                                use_cache = False)

        generated = tokenizer.decode(outputs[0],
                                    skip_special_tokens = True
            ).strip()

        cut = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:],
                                    skip_special_tokens = True
            ).strip()
        ###
        print(cut)
        test['sentence'].append(dataset['sentence'][i])
        test['full_output'].append(generated)
        test['output'].append(cut)
        test['label'].append(data['output'][i])
      test = pd.DataFrame(test)
      test.to_csv(path+'results/'+model_type+'/'+compound_or_vmwe+'_train_'+language+'_test_'+test_language+'_'+str(d)+'.csv')
      del model
      del tokenizer
      gc.collect()
      torch.cuda.empty_cache()
      torch.cuda.ipc_collect()


In [5]:
inference(0,1,'llama-8b-base','compound_0shot','na','ZH')
inference(0,1,'llama-8b-base','compound_fewshot','na','ZH')
inference(0,1,'llama-8b-base','vmwe_0shot','na','ZH')
inference(0,1,'llama-8b-base','vmwe_fewshot','na','ZH')


Filter:   0%|          | 0/1215 [00:00<?, ? examples/s]

Filter:   0%|          | 0/1215 [00:00<?, ? examples/s]

1964年1月28日(1964-01-28)(52歲)
==((====))==  Unsloth 2026.4.2: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    NVIDIA RTX PRO 6000 Blackwell Server Edition. Num GPUs = 1. Max memory: 94.971 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 12.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

...

No MWE found.
No MWE found.


Both `max_new_tokens` (=100) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=100) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


No MWE found.
No MWE found.


Both `max_new_tokens` (=100) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=100) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


No MWE found.
No MWE found.
No MWE found.
